# Unwrapping AI — Step 06: Regularization
## A beginner-to-practitioner learning book in Python

**Audience:** Someone who has never studied regularization before.

**What this book will teach:** Why machine-learning models overfit, how regularization works, what Ridge and Lasso do, how to implement them in Python, and how to interpret the results.

### How to use this notebook
- Read the explanation before running each code cell.
- Run cells from top to bottom the first time.
- If you return to a section later, read its explanation and run its code cell; important variables are re-created where needed.
- Do not judge a model only by its training score. We care about performance on data the model has not seen.

**Learning path:** intuition → mathematics → Python foundations → data → baseline model → Ridge → Lasso → comparison → tuning → business interpretation → exercises.

## Contents
1. The problem regularization solves
2. Essential vocabulary
3. A simple analogy
4. The mathematics, explained symbol by symbol
5. Python syntax and operators used in this lesson
6. Load and understand the dataset
7. Split data correctly
8. Build a baseline model
9. Understand Ridge (L2)
10. Understand Lasso (L1)
11. Compare model performance
12. Understand alpha and coefficient shrinkage
13. Common mistakes and troubleshooting
14. Business example and interpretation
15. Exercises with solutions
16. Glossary and final checklist

# 1. The problem regularization solves

Imagine teaching a student using 100 practice questions. The student memorises every answer but does not understand the subject. When given a new exam, the student performs poorly.

A machine-learning model can behave in the same way.

- **Training data:** examples used to teach the model.
- **Test data:** separate examples used to check how well it learned.
- **Generalization:** the ability to perform well on new, unseen examples.
- **Overfitting:** when a model learns training-specific details, including noise, rather than the underlying pattern.

Regularization is a family of techniques that discourages a model from becoming unnecessarily complex. In linear models, Ridge and Lasso do this by adding a penalty related to the size of the model's coefficients.

**Important:** Regularization does not guarantee a better model. Too much regularization can make a model too simple. We must evaluate it on data that was not used to fit it.

## 2. Essential vocabulary

| Term | Plain-English meaning |
|---|---|
| Feature | An input variable used to make a prediction |
| Target | The value we want to predict |
| Observation / row | One example in the dataset |
| Model | A mathematical method that learns patterns |
| Coefficient / weight | A number that controls a feature's influence |
| Intercept / bias | The model's starting prediction when features are zero |
| Prediction | The value produced by a model |
| Residual / error | Actual value minus predicted value |
| Loss | A numerical measure of prediction error |
| MSE | Mean Squared Error; average of squared prediction errors |
| Penalty | An extra cost added to discourage large coefficients |
| Alpha | A setting that controls penalty strength |
| Scaling | Transforming features to comparable numerical ranges |
| Validation | A process for choosing model settings |
| Test set | Data held back for final evaluation |

## 3. A simple analogy: fitting a curve

Think of drawing a line through points on a graph. A straight line may miss some individual points but capture the overall trend. A very complicated curve might pass close to every training point, including random noise.

Regularization asks the model not to use unnecessarily large coefficients to chase every small detail.

- No penalty: the model focuses on minimising prediction error.
- A penalty: the model balances prediction error with coefficient size.
- Excessive penalty: the model may become too restricted and underfit.

This is the balance between fitting the data and generalising to new data.

# 4. The mathematics, explained symbol by symbol

## 4.1 Linear regression prediction

For one feature:

\[
\hat{y} = wx + b
\]

- \(\hat{y}\): predicted value (read “y-hat”)
- \(x\): input feature
- \(w\): coefficient or weight
- \(b\): intercept or bias

For several features:

\[
\hat{y} = w_1x_1+w_2x_2+\dots+w_px_p+b
\]

Each feature has a coefficient.

## 4.2 Mean Squared Error

\[
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
\]

- \(n\): number of observations
- \(i\): position of the observation
- \(y_i\): actual target for observation \(i\)
- \(\hat{y}_i\): predicted target
- \(\sum\): add the values for all observations
- squared: multiply the error by itself
- \(1/n\): divide the total by the number of observations

MSE measures prediction error. A smaller MSE is generally better when comparing predictions on the same target scale.

## 4.3 Ridge regression — L2 penalty

\[
Objective = MSE + \alpha\sum_{j=1}^{p}w_j^2
\]

The penalty adds the squared coefficients.

- \(p\): number of features
- \(j\): feature position
- \(w_j\): coefficient for feature \(j\)
- \(w_j^2\): coefficient multiplied by itself
- \(\alpha\): strength of the penalty

Ridge usually shrinks coefficients towards zero. It generally does not make them exactly zero.

## 4.4 Lasso regression — L1 penalty

\[
Objective = MSE + \alpha\sum_{j=1}^{p}|w_j|
\]

The vertical bars mean absolute value. For example, the absolute value of -5 is 5.

Lasso can make some coefficients exactly zero. This can create a simpler model with fewer active features.

**A note on implementation:** Scikit-learn's Ridge and Lasso use specific objective-function conventions and scaling factors. The formulas above communicate the core idea; do not assume every library writes the loss with exactly the same constant factors.

# 5. Python syntax and operators used in this lesson

You do not need to memorise these. Understand what each one means.

| Syntax / operator | Meaning | Example |
|---|---|---|
| `=` | Assign a value to a variable | `alpha = 1.0` |
| `()` | Call a function or group an expression | `print(alpha)` |
| `[]` | Access an item or create a list | `names[0]` |
| `.` | Access an object's method or attribute | `model.fit()` |
| `,` | Separate function arguments or list items | `train_test_split(X, y)` |
| `:` | Start an indented code block or define a slice | `for x in items:` |
| `+` | Addition | `2 + 3` |
| `-` | Subtraction or negative sign | `5 - 2` |
| `*` | Multiplication | `3 * 4` |
| `/` | Division | `10 / 2` |
| `**` | Exponent / power | `3 ** 2` means 9 |
| `==` | Test whether two values are equal | `x == 5` |
| `!=` | Test whether two values differ | `x != 5` |
| `<`, `>` | Less than, greater than | `score > 0.8` |
| `#` | Comment; Python does not execute it | `# explanation` |

**Important distinction:** `=` assigns a value. `==` compares two values. They are not interchangeable.

A Python function may receive **arguments** inside parentheses. For example, `Ridge(alpha=1.0)` calls the Ridge function and supplies a named argument called `alpha`.

The dot in `model.fit(X_train, y_train)` accesses the `fit` method belonging to the model object. The method trains that model.

In [ ]:
# A tiny Python syntax demonstration
alpha = 1.0          # '=' assigns the number 1.0 to alpha
squared = alpha ** 2 # '**' raises alpha to the power of 2
half = alpha / 2     # '/' divides alpha by 2
is_equal = alpha == 1.0  # '==' checks equality

print("alpha:", alpha)
print("alpha squared:", squared)
print("half of alpha:", half)
print("Is alpha equal to 1.0?", is_equal)

**What should you see?** The output should show 1.0, 1.0, 0.5 and True.

`print()` displays information. Text inside quotation marks is a string. Commas let `print()` display multiple items. The `#` symbol starts a comment; Python ignores the comment when executing the cell.

# 6. Load and understand the dataset

We will use the California Housing dataset, a commonly used teaching dataset. Each row describes a geographic area, with features such as median income and average rooms. The target is a housing-value measure.

We use this dataset to demonstrate the workflow, not to build a real property valuation service.

**Why use a real dataset?** Regularization is easier to understand when features have different scales and the model has several coefficients.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download/load the dataset through scikit-learn.
housing = fetch_california_housing()

# Put the input features into a labelled table.
X = pd.DataFrame(housing.data, columns=housing.feature_names)

# The target is the value the model will learn to predict.
y = pd.Series(housing.target, name="Target")

print("Feature table shape:", X.shape)
print("Target shape:", y.shape)
print("\nFirst five feature rows:")
display(X.head())
print("\nFirst five target values:")
display(y.head())

### Understand the code above

- `fetch_california_housing()` loads the example dataset.
- `housing.data` contains the input feature values.
- `housing.feature_names` contains the feature names.
- `pd.DataFrame(...)` creates a table with named columns.
- `housing.target` contains the target values.
- `pd.Series(...)` creates a labelled one-dimensional data structure.
- `.shape` reports dimensions: rows first, columns second.
- `.head()` displays the first five rows by default.
- `display()` renders a table neatly in a notebook.

The first run may need an internet connection to download the dataset if it is not cached. If the download fails, check your internet connection and rerun this cell.

In [ ]:
# Inspect basic information and missing values
print("Dataset information:")
X.info()

print("\nMissing values in each feature:")
display(X.isna().sum())

print("\nSummary statistics:")
display(X.describe().round(2))

`X.info()` reports column types and non-null counts. `X.isna()` marks missing values; `.sum()` counts them column by column. `.describe()` gives summary statistics such as mean, standard deviation, minimum and maximum.

Why check this? A model cannot be responsibly trained without understanding the data. Missing values, unusual scales and unexpected data types can affect the results.

# 7. Split data correctly

We separate the dataset into training and test portions.

- Training data teaches the model.
- Test data estimates how well the final model performs on unseen examples.

We must not use the test set to repeatedly choose alpha. If we do, we indirectly tune to the test set and the final test score becomes overly optimistic. Later, cross-validation will help us choose settings using training data only.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,                  # input features
    y,                  # target values
    test_size=0.20,     # reserve 20% for testing
    random_state=42     # make the split reproducible
)

print("Training feature rows:", X_train.shape[0])
print("Testing feature rows:", X_test.shape[0])
print("Training target rows:", y_train.shape[0])
print("Testing target rows:", y_test.shape[0])

### Understand each argument

`train_test_split()` divides aligned features and targets into training and test sets.

- `X` and `y` must correspond row by row.
- `test_size=0.20` reserves 20% for testing; the remaining 80% is training data.
- `random_state=42` fixes the random split so another run produces the same split.
- `X_train`, `X_test`, `y_train`, `y_test` are four outputs returned by the function.

The `.shape[0]` expression means: read the shape tuple and select its first item, the number of rows.

# 8. Build a baseline: ordinary Linear Regression

A baseline gives us a reference point. We first train a model without Ridge or Lasso penalties.

We will use a pipeline with `StandardScaler`. Scaling transforms each feature so its values are centred around zero and measured in standard-deviation units.

**Why scale?** Regularization penalises coefficient sizes. If one feature is measured in thousands and another in fractions, their coefficient sizes can be affected by units. Scaling makes the penalty more comparable across features.

The scaler is fitted on training data only because the pipeline is trained with `X_train`. It then applies the learned transformation to test data.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

baseline = make_pipeline(
    StandardScaler(),  # learn scaling from training data
    LinearRegression() # fit a linear regression model
)

baseline.fit(X_train, y_train)  # learn from training examples
baseline_predictions = baseline.predict(X_test)  # predict unseen test rows

baseline_mse = mean_squared_error(y_test, baseline_predictions)
baseline_rmse = np.sqrt(baseline_mse)
baseline_mae = mean_absolute_error(y_test, baseline_predictions)
baseline_r2 = r2_score(y_test, baseline_predictions)

print("Baseline Linear Regression")
print("MSE: ", round(baseline_mse, 4))
print("RMSE:", round(baseline_rmse, 4))
print("MAE: ", round(baseline_mae, 4))
print("R²:  ", round(baseline_r2, 4))

### What do these metrics mean?

- **MSE:** average squared prediction error. Large errors receive a stronger penalty because errors are squared.
- **RMSE:** square root of MSE. It is expressed in the target's original units.
- **MAE:** average absolute prediction error. It is also in the target's original units.
- **R²:** a measure of how much variation in the target is explained by the model relative to a mean-prediction baseline. It can be negative on test data.

`fit()` learns the model. `predict()` makes predictions. `round(value, 4)` displays a number rounded to four decimal places.

Do not compare metrics from different target scales without considering their units.

# 9. Ridge regression (L2): shrink coefficients

Ridge adds a penalty based on squared coefficients. The model tries to keep prediction errors low while avoiding unnecessarily large weights.

A larger `alpha` means a stronger penalty. Ridge generally shrinks coefficients towards zero, but does not usually remove features by making coefficients exactly zero.

We will start with alpha = 1.0. This is a demonstration value, not a claim that 1.0 is optimal.

In [ ]:
from sklearn.linear_model import Ridge

ridge_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

ridge_model.fit(X_train, y_train)
ridge_predictions = ridge_model.predict(X_test)

ridge_mse = mean_squared_error(y_test, ridge_predictions)
ridge_rmse = np.sqrt(ridge_mse)
ridge_r2 = r2_score(y_test, ridge_predictions)

print("Ridge Regression (alpha=1.0)")
print("Test MSE: ", round(ridge_mse, 4))
print("Test RMSE:", round(ridge_rmse, 4))
print("Test R²:  ", round(ridge_r2, 4))

### Explain the code

- `Ridge` is the Scikit-learn estimator for Ridge regression.
- `alpha=1.0` sets the penalty strength.
- `make_pipeline(...)` chains scaling and model fitting into one workflow.
- `ridge_model.fit(...)` fits both pipeline stages using training data.
- `ridge_model.predict(...)` scales the test features using the training-fitted scaler and then predicts.
- The metrics quantify performance on the test set.

A result better than the baseline is possible, but not guaranteed. The purpose is to learn how to compare fairly.

# 10. Lasso regression (L1): shrink and potentially remove coefficients

Lasso adds a penalty based on the absolute values of coefficients.

Unlike Ridge, Lasso can shrink some coefficients exactly to zero. A zero coefficient means that feature is not contributing to the fitted linear prediction in that model.

This is sometimes called embedded feature selection. It does not mean Lasso has discovered the universally unimportant features; the result depends on the dataset, scaling, alpha and other correlated features.

In [ ]:
from sklearn.linear_model import Lasso

lasso_model = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.01, max_iter=20000)
)

lasso_model.fit(X_train, y_train)
lasso_predictions = lasso_model.predict(X_test)

lasso_mse = mean_squared_error(y_test, lasso_predictions)
lasso_rmse = np.sqrt(lasso_mse)
lasso_r2 = r2_score(y_test, lasso_predictions)

print("Lasso Regression (alpha=0.01)")
print("Test MSE: ", round(lasso_mse, 4))
print("Test RMSE:", round(lasso_rmse, 4))
print("Test R²:  ", round(lasso_r2, 4))

`max_iter=20000` allows the optimisation routine many iterations to converge. If you see a convergence warning, it means the optimiser did not meet its stopping condition within the allowed iterations. Scaling helps; increasing `max_iter` may help, but warnings should be investigated rather than ignored.

# 11. Compare the three models

Now we put the metrics side by side. This is more useful than looking at one model's score in isolation.

Remember: the models used the same training/test split and the same test examples, so the comparison is meaningful.

In [ ]:
comparison = pd.DataFrame([
    {"Model": "Linear Regression", "Test MSE": baseline_mse,
     "Test RMSE": baseline_rmse, "Test R2": baseline_r2},
    {"Model": "Ridge (alpha=1.0)", "Test MSE": ridge_mse,
     "Test RMSE": ridge_rmse, "Test R2": ridge_r2},
    {"Model": "Lasso (alpha=0.01)", "Test MSE": lasso_mse,
     "Test RMSE": lasso_rmse, "Test R2": lasso_r2}
])

display(comparison.round(4))

**How to interpret:** Lower MSE and RMSE are preferable; higher R² is preferable. But small differences may not be meaningful, and a single split can be affected by chance. Cross-validation gives a more robust way to compare settings.

# 12. Understand alpha: the strength of regularization

Alpha is not a measure of model accuracy. It controls how strongly the penalty influences coefficient values.

- Very small alpha: penalty is weak; Ridge approaches ordinary linear regression as alpha approaches zero.
- Larger alpha: stronger shrinkage.
- Extremely large alpha: coefficients can become too small, and the model may underfit.

We will try several alpha values for Ridge and measure test performance. This is an illustration only. For real model selection, use cross-validation on the training data and keep the test set for final evaluation.

In [ ]:
alpha_values = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_alpha_results = []

for alpha_value in alpha_values:
    candidate = make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha_value)
    )
    candidate.fit(X_train, y_train)
    candidate_pred = candidate.predict(X_test)

    ridge_alpha_results.append({
        "alpha": alpha_value,
        "test_mse": mean_squared_error(y_test, candidate_pred),
        "test_r2": r2_score(y_test, candidate_pred)
    })

ridge_alpha_df = pd.DataFrame(ridge_alpha_results)
display(ridge_alpha_df.round(4))

### Understand the loop

- `alpha_values` is a list of candidate penalty strengths.
- `for alpha_value in alpha_values:` means repeat the indented instructions once for each value.
- Each iteration creates a fresh Ridge pipeline.
- The candidate is fitted on the training data.
- It predicts the test data and calculates metrics.
- `.append(...)` adds a dictionary of results to the results list.
- `pd.DataFrame(...)` turns that list of dictionaries into a table.

The indentation is important: Python knows which instructions belong inside the loop because they are indented.

In [ ]:
plt.plot(
    ridge_alpha_df["alpha"],
    ridge_alpha_df["test_mse"],
    marker="o"
)
plt.xscale("log")
plt.xlabel("Alpha (logarithmic scale)")
plt.ylabel("Test MSE")
plt.title("Ridge: penalty strength and test error")
plt.grid(True)
plt.show()

The x-axis uses a logarithmic scale because the alpha values span several powers of ten. This helps us see small and large values on one chart. The chart shows how test error changed for this split; it does not establish a universal best alpha.

# 13. Look at coefficient shrinkage

A coefficient represents the model's learned relationship between a scaled feature and the prediction, holding other features fixed.

We will extract Ridge coefficients for weak, moderate and strong penalties and compare their magnitudes.

The pipeline stores its steps with names. `named_steps["ridge"]` retrieves the Ridge estimator inside the pipeline. `.coef_` is a fitted attribute containing the learned coefficients.

In [ ]:
alphas_to_inspect = [0.01, 1, 100]
coefficient_table = pd.DataFrame(index=X.columns)

for alpha_value in alphas_to_inspect:
    candidate = make_pipeline(
        StandardScaler(),
        Ridge(alpha=alpha_value)
    )
    candidate.fit(X_train, y_train)

    fitted_ridge = candidate.named_steps["ridge"]
    coefficient_table[f"alpha={alpha_value}"] = fitted_ridge.coef_

display(coefficient_table.round(4))

In [ ]:
coefficient_table.plot(kind="bar", figsize=(12, 5))
plt.ylabel("Coefficient value")
plt.title("Ridge coefficients at different alpha values")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

As alpha increases, coefficient magnitudes generally shrink. Do not interpret the coefficient sign or size without remembering that the features were standardised. Coefficients are now associated with standardised feature changes, not one raw unit of the original feature.

# 14. Observe Lasso's zero coefficients

Now we inspect the Lasso coefficients. A coefficient that is exactly zero indicates that the fitted Lasso model has removed that feature's direct linear contribution.

Correlated features can affect which feature Lasso retains. Different data splits or alpha values can produce different selections.

In [ ]:
lasso_for_features = make_pipeline(
    StandardScaler(),
    Lasso(alpha=0.05, max_iter=20000)
)
lasso_for_features.fit(X_train, y_train)

lasso_coefficients = lasso_for_features.named_steps["lasso"].coef_

lasso_feature_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lasso_coefficients,
    "Coefficient is zero": np.isclose(lasso_coefficients, 0)
})

display(lasso_feature_table.round(4))
print("Number of zero coefficients:",
      int(np.isclose(lasso_coefficients, 0).sum()))

`np.isclose(value, 0)` checks whether a floating-point number is sufficiently close to zero. It is safer than relying on exact equality for many numerical calculations.

`int(...)` converts the count into a standard integer for display.

# 15. Common mistakes and troubleshooting

**Mistake 1 — Forgetting to scale features**

Regularization depends on coefficient size. Features measured on very different scales can be penalised unfairly. Use a pipeline with `StandardScaler` for this workflow.

**Mistake 2 — Scaling before the train/test split**

If you fit a scaler on the entire dataset before splitting, information from the test set influences preprocessing. Split first, then fit preprocessing on training data. A Scikit-learn pipeline helps prevent this leakage.

**Mistake 3 — Assuming larger alpha is always better**

A stronger penalty can reduce overfitting, but too much penalty can cause underfitting. Evaluate candidate values with cross-validation.

**Mistake 4 — Choosing alpha using the final test set**

Repeatedly checking the test score and selecting the best alpha leaks information from the test set into model selection. Use cross-validation on training data instead.

**Mistake 5 — Thinking Ridge removes features**

Ridge generally shrinks coefficients but does not usually make them exactly zero. Lasso can produce exact zeros.

**Mistake 6 — Comparing models on different splits**

Use the same split and evaluation metrics when comparing models.

**Mistake 7 — Treating one score as the whole story**

Look at multiple metrics, the data, model stability, and the real-world cost of errors.

## Troubleshooting guide

| Symptom | Possible reason | What to try |
|---|---|---|
| Dataset download error | Internet or source unavailable | Check connection and rerun load cell |
| `NameError` | Earlier cell not run, or variable name typo | Run prerequisite cells from the top |
| Convergence warning | Optimiser needs more iterations or different settings | Scale features; increase `max_iter`; inspect alpha |
| Unexpectedly poor score | Data quality, split variation, model limitations | Inspect data and compare a baseline |
| Table not displayed | `display` not imported in a different environment | Run `from IPython.display import display` |
| Plot does not appear | Plot cell not run or plotting issue | Run cell again; check Matplotlib installation |

# 16. Industry example: predicting loan default risk

Imagine a bank has a model that predicts the probability of a customer defaulting on a loan.

Possible features include:
- Customer income
- Existing debt
- Credit utilisation
- Number of missed payments
- Length of credit history

A model with very large coefficients may be overly sensitive to small changes in some inputs. Regularization can help control coefficient size and improve stability.

However:
- A regularized model is not automatically fair or compliant.
- Feature selection does not prove that a feature is safe or appropriate to use.
- Sensitive decisions need validation, explainability, governance and human oversight.
- The best alpha must be selected using a sound validation process.

**Business lesson:** Regularization is one tool for building models that generalise. It does not replace domain knowledge, data quality checks, fairness analysis or monitoring.

# 17. Exercises

Try these before opening the solutions.

**Exercise 1:** In your own words, explain why a regularization penalty is added to the loss function.

**Exercise 2:** Which method can make some coefficients exactly zero: Ridge or Lasso?

**Exercise 3:** What does alpha control?

**Exercise 4:** Why should scaling be included in a regularized modelling pipeline?

**Exercise 5:** Why should alpha be selected using cross-validation rather than the final test set?

**Exercise 6 — Practical:** Change the Lasso alpha from `0.05` to `0.5`. Run the Lasso coefficient cell again. How does the number of zero coefficients change? What happens to test performance if you also evaluate this model?

**Exercise 7 — Reflection:** If Ridge has a slightly lower test MSE than Linear Regression on one split, can you conclude Ridge is always better? Explain.

# 18. Exercise solutions

**1.** The penalty discourages unnecessarily large coefficients, balancing fit to training data with model complexity.

**2.** Lasso can make some coefficients exactly zero. Ridge usually shrinks coefficients towards zero without setting them exactly to zero.

**3.** Alpha controls the strength of the regularization penalty. A larger alpha means a stronger penalty.

**4.** Scaling puts features on comparable scales so the penalty does not unfairly depend on the units used to measure a feature.

**5.** Cross-validation estimates performance across multiple training/validation splits. The final test set should remain untouched until final evaluation.

**6.** A larger Lasso alpha usually increases shrinkage and may create more zero coefficients. Test performance may improve or worsen; measure it rather than guessing.

**7.** No. One split can be affected by chance. Use cross-validation and a final held-out test set to make a more reliable comparison.

# 19. Glossary

- **Alpha:** regularization strength.
- **Baseline:** a simple reference model used for comparison.
- **Coefficient:** learned weight attached to a feature.
- **Cross-validation:** repeated training/validation splits used to assess model settings.
- **Feature:** input variable.
- **Generalization:** performance on new data.
- **Intercept:** model's constant term.
- **L1 penalty:** sum of absolute coefficient values; used by Lasso.
- **L2 penalty:** sum of squared coefficient values; used by Ridge.
- **Lasso:** linear regression with an L1 penalty.
- **Loss function:** numerical measure the model seeks to minimise.
- **Overfitting:** fitting training-specific noise or details.
- **Regularization:** techniques that discourage excessive model complexity.
- **Ridge:** linear regression with an L2 penalty.
- **Scaling:** transforming features to comparable ranges.
- **Target:** value the model predicts.
- **Training set:** data used to fit the model.
- **Test set:** held-out data used for final evaluation.
- **Underfitting:** model is too simple to capture the pattern.

# 20. Final revision and completion checklist

You have completed this chapter when you can explain, without copying:
- [ ] Why overfitting is a problem.
- [ ] What regularization adds to the objective function.
- [ ] The difference between L1 and L2 penalties.
- [ ] Why Ridge shrinks coefficients.
- [ ] Why Lasso can set coefficients to zero.
- [ ] What alpha controls.
- [ ] Why feature scaling matters.
- [ ] Why the test set must be kept separate during model selection.
- [ ] How to compare models using MSE, RMSE and R².

## What comes next?

**Step 07 — Cross-Validation**

You will learn how to evaluate models more reliably, use K-Fold Cross-Validation, compare model settings, and choose hyperparameters without repeatedly relying on the final test set.

**Unwrapping AI — Learn the concept. Understand the code. Build the skill.**